# MLflow connection examples

This notebook demonstrates how to connect a Jupyter notebook to an MLflow tracking server and log experiment results.

MLflow is useful for keeping track of machine learning experiments. Instead of manually writing down which parameters were used and what results were achieved, we can log this information automatically or manually during the notebook run.

In this example, we will:

- configure the MLflow tracking URI,
- create or select an experiment,
- log a simple metric,
- train a small scikit-learn model,
- log model parameters and evaluation metrics,
- register the trained model in MLflow.

In [ ]:
# import 
import gc
import os
import os.path
import sys
import mlflow
from pathlib import Path

## Configure MLflow tracking

Before we can log anything, the notebook needs to know where the MLflow tracking backend is running.

The tracking URI defines where MLflow stores information about experiments, runs, metrics, parameters, tags, artifacts, and models. In this environment, the backend store path is set explicitly or via environment variable.

In [ ]:
# setting tracking uri
mlflow.set_tracking_uri("http://mlflow")

In [ ]:
experiment_name = "test-experiment"

artifact_location = Path.home() / "bucket" / experiment_name
artifact_location.mkdir(parents=True, exist_ok=True)

experiment = mlflow.get_experiment_by_name(experiment_name)

if experiment is None:
    mlflow.create_experiment(
        name=experiment_name,
        artifact_location=artifact_location.as_uri(),
    )
    print("Created experiment:", experiment_name)
else:
    print("Experiment already exists:", experiment_name)

mlflow.set_experiment(experiment_name)


## Log a first test metric

This first example creates a new MLflow run and logs one simple metric.

A **run** represents one execution of an experiment. During a run, we can log metrics, parameters, tags, artifacts, and models. This small test is useful for checking that the MLflow connection works before running a real machine learning example.

In [ ]:
# basic logging of values into experiment
# data can be seen in the MLflow URL: https://hub.eox.at/services/eoxhub-gateway/orbitalai1/mlflow/ 
with mlflow.start_run() as run:
    mlflow.log_metric(key="metric1", value=1.0)


## Automatic logging

MLflow supports automatic logging for several machine learning libraries. With autologging enabled, MLflow can automatically capture useful information such as model parameters, training metrics, and model artifacts.

Autologging is convenient when we want quick experiment tracking without adding many explicit `mlflow.log_*` calls to the training code.

Using autolog() function from appropriate flavor corresponding to code will automatically log all parameters and metrics available. Many flavors support adding log_models=True argument to automatically log model. 

List of supported flavours and the documentation can be found here: https://mlflow.org/docs/latest/tracking.html#automatic-logging

In [ ]:
!pip3 install tensorflow

## TensorFlow example

This example trains a small neural network using TensorFlow/Keras and logs the training process with MLflow autologging.

The model uses the Reuters dataset, which is a small text classification dataset included with Keras. The goal of this example is not to build the best possible model, but to demonstrate how MLflow can capture training information from a TensorFlow workflow.

In [ ]:
import numpy as np
import mlflow
from tensorflow import keras
from tensorflow.keras.datasets import reuters
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout, Activation
from tensorflow.keras.preprocessing.text import Tokenizer

# Start MLflow TensorFlow autologging
mlflow.tensorflow.autolog()

max_words = 1000
batch_size = 32
epochs = 5

print("Loading data...")
(x_train, y_train), (x_test, y_test) = reuters.load_data(
    num_words=max_words,
    test_split=0.2,
)

print(len(x_train), "train sequences")
print(len(x_test), "test sequences")

# Cast to plain Python int for newer Keras compatibility
num_classes = int(np.max(y_train) + 1)
print(num_classes, "classes")

print("Vectorizing sequence data...")
tokenizer = Tokenizer(num_words=max_words)
x_train = tokenizer.sequences_to_matrix(x_train, mode="binary")
x_test = tokenizer.sequences_to_matrix(x_test, mode="binary")

print("x_train shape:", x_train.shape)
print("x_test shape:", x_test.shape)

print("Convert class vector to binary class matrix")
y_train = keras.utils.to_categorical(y_train, num_classes)
y_test = keras.utils.to_categorical(y_test, num_classes)

print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("Building model...")
model = Sequential()
model.add(Input(shape=(max_words,)))
model.add(Dense(512))
model.add(Activation("relu"))
model.add(Dropout(0.5))
model.add(Dense(num_classes))
model.add(Activation("softmax"))

model.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)

history = model.fit(
    x_train,
    y_train,
    batch_size=batch_size,
    epochs=epochs,
    verbose=1,
    validation_split=0.1,
)

score = model.evaluate(
    x_test,
    y_test,
    batch_size=batch_size,
    verbose=1,
)

print("Test score:", score[0])
print("Test accuracy:", score[1])

## Manual logging with scikit-learn

In this section, we train a simple scikit-learn model and manually decide what to log to MLflow.

Manual logging gives us more control than autologging. We explicitly log the parameters we care about, such as `alpha` and `l1_ratio`, and the evaluation metrics we want to compare between runs, such as RMSE, MAE, and R².

Almost any code can be wrapped into with ```mlflow.start_run() as run:``` and manual logging of selected parameters can be done using:
    ```mlflow.log_params()``` or 
    ```mlflow.log_metric()```

In [ ]:
import os
import warnings
import sys

import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import ElasticNet
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn

        

In [ ]:
def eval_metrics(actual, pred):
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mae = mean_absolute_error(actual, pred)
    r2 = r2_score(actual, pred)
    return rmse, mae, r2

In [ ]:
csv_url = (
    "https://raw.githubusercontent.com/mlflow/mlflow/master/tests/datasets/winequality-red.csv"
)
try:
    data = pd.read_csv(csv_url, sep=";")
except Exception as e:
    pass

# Split the data into training and test sets. (0.75, 0.25) split.
train, test = train_test_split(data)

# The predicted column is "quality" which is a scalar from [3, 9]
train_x = train.drop(["quality"], axis=1)
test_x = test.drop(["quality"], axis=1)
train_y = train[["quality"]]
test_y = test[["quality"]]


alpha = 0.5
l1_ratio = 0.5

with mlflow.start_run():
    lr = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, random_state=42)
    lr.fit(train_x, train_y)

    predicted_qualities = lr.predict(test_x)

    rmse, mae, r2 = eval_metrics(test_y, predicted_qualities)

    print(f"Elasticnet model (alpha={alpha:f}, l1_ratio={l1_ratio:f}):")
    print(f"  RMSE: {rmse}")
    print(f"  MAE: {mae}")
    print(f"  R2: {r2}")

    mlflow.log_param("alpha", alpha)
    mlflow.log_param("l1_ratio", l1_ratio)

    mlflow.set_tag("note", "tyna was here")

    mlflow.log_metric("rmse", float(rmse))
    mlflow.log_metric("r2", float(r2))
    mlflow.log_metric("mae", float(mae))

    mlflow.sklearn.log_model(
        sk_model=lr,
        name="model",
    )


## Inspect the results in MLflow

After running the notebook, open the MLflow user interface and inspect the experiment.

You should see a new run containing:

- logged parameters such as `alpha` and `l1_ratio`,
- logged metrics such as `rmse`, `mae`, and `r2`,
- optional tags with descriptive metadata,
- the trained scikit-learn model artifact.

This makes it easier to compare different model runs and understand which settings produced better results.